In [33]:
import pandas as pd
import os
from sklearn.linear_model import LogisticRegressionCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, GridSearchCV, ParameterGrid
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import recall_score, make_scorer, precision_score
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.base import clone
from sklearn.metrics import recall_score
df_model = pd.read_csv(r'../data/raw_churn_data.csv')

In [2]:
df_model.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [3]:
df_model['TotalCharges'] = pd.to_numeric(df_model['TotalCharges'], errors='coerce')
df_model.loc[df_model['tenure'] == 0, 'TotalCharges'] = 0
X, y = df_model.drop(columns=['Churn', 'customerID']), df_model['Churn']
num_cols = X.select_dtypes(include=['int64','float64']).columns
cat_cols = X.select_dtypes(include=['object']).columns
print(num_cols ,'\n')
print(cat_cols)

Index(['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges'], dtype='object') 

Index(['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
       'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
       'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract',
       'PaperlessBilling', 'PaymentMethod'],
      dtype='object')


In [4]:
# look at the sklearn ColumnTransformer or Pipeline documentation for review
num_pipe = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())])
cat_pipe = Pipeline([('imputer', SimpleImputer(strategy='most_frequent')),('ohe', (OneHotEncoder(handle_unknown='ignore')))])


preprocessor = ColumnTransformer([('num', num_pipe, num_cols), ('cat', cat_pipe, cat_cols)])

In [5]:
# X,y were identified in cell 2
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Second split: 75% train - 25% validation for XGBoost early stopping (60% train, 20% val, 20% test overall)
X_train_xgb, X_val, y_train_xgb, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=42, stratify=y_train)


In [6]:
# Using LogisticRegressionCV for AUTO hyperparameter tuning with *Stratified K-Folds* CV instead of KFold because its better for classification
logreg_pipe = Pipeline([('preprocess', preprocessor), ('logregcv', LogisticRegressionCV(max_iter=2000, scoring='recall', cv=10))])

best_logreg = logreg_pipe.fit(X_train, y_train)

y_pred = best_logreg.predict(X_test)
from sklearn.metrics import classification_report, confusion_matrix
print('Logistic Regression CV\n')
print('Best C:', best_logreg.named_steps['logregcv'].C_, '\n')
print('Confusion Matrix:\n', confusion_matrix(y_test, y_pred))
print('\nClassification Report:\n', classification_report(y_test, y_pred))

Logistic Regression CV

Best C: [2.7825594] 

Confusion Matrix:
 [[926 109]
 [163 211]]

Classification Report:
               precision    recall  f1-score   support

          No       0.85      0.89      0.87      1035
         Yes       0.66      0.56      0.61       374

    accuracy                           0.81      1409
   macro avg       0.75      0.73      0.74      1409
weighted avg       0.80      0.81      0.80      1409



In [9]:
rf = RandomForestClassifier(n_estimators= 300, oob_score=True, random_state=42, class_weight='balanced', n_jobs=-1)

rf_pipe = Pipeline([('preprocessor', preprocessor), ('rf', rf)])

param_grid = {
    'rf__max_depth': [6,8,10,None],
    'rf__min_samples_leaf': [5, 10, 20, 40]
}

# pos/neg classes are defaulted at 1/0 - changed to yes/no  
recall_scorer = make_scorer(recall_score, pos_label='Yes')

rf_cv = GridSearchCV( rf_pipe, param_grid, scoring= recall_scorer, cv=10 , n_jobs=-1)

rf_cv.fit(X_train, y_train)
best_rf = rf_cv.best_estimator_
y_pred = best_rf.predict(X_test)

print('Best params:', rf_cv.best_params_)
print('Best recall (train):', rf_cv.best_score_)

print('Random Forest\n')
print('Confusion Matrix: \n', confusion_matrix(y_test, y_pred))
print('Classification Report: \n', classification_report(y_test, y_pred))

Best params: {'rf__max_depth': 6, 'rf__min_samples_leaf': 10}
Best recall (train): 0.8013288590604027
Random Forest

Confusion Matrix: 
 [[762 273]
 [ 77 297]]
Classification Report: 
               precision    recall  f1-score   support

          No       0.91      0.74      0.81      1035
         Yes       0.52      0.79      0.63       374

    accuracy                           0.75      1409
   macro avg       0.71      0.77      0.72      1409
weighted avg       0.81      0.75      0.76      1409



In [7]:
# 75% train (x_train_xgb) - 25% validation (x_val) for XGBoost early stopping 
# (60% train, 20% val, 20% test overall)

# Made yes and no numeric for xgboost (required)
y_train_xgb_numeric = y_train_xgb.map({'No': 0, 'Yes': 1})
y_val_numeric = y_val.map({'No': 0, 'Yes': 1})
y_test_numeric = y_test.map({'No': 0, 'Yes': 1})

# Fit [clone of] preprocessor ONLY on XGBoost training data, then (impute, ohe, scale) applied
preprocessor_xgb = clone(preprocessor).fit(X_train_xgb)

X_train_processed = preprocessor_xgb.transform(X_train_xgb)
X_val_processed = preprocessor_xgb.transform(X_val)
X_test_processed = preprocessor_xgb.transform(X_test)

# early stopping to reduce overfitting


param_grid_xgb = {
    'max_depth': [4, 5, 6],
    'learning_rate': [0.05, 0.1, 0.15]
}

best_pram = None
best_xgb = None
best_recall = -1
for pram in ParameterGrid(param_grid_xgb):
    xgb = XGBClassifier(
    objective='binary:logistic',
    eval_metric='aucpr',
    n_estimators=500,
    random_state=42,
    scale_pos_weight=(y_train_xgb == 'No').sum() / (y_train_xgb == 'Yes').sum(),
    early_stopping_rounds=25, **pram
    )
    # each iteration fits different parameters (9 models)
    xgb.fit(
        X_train_processed, 
        y_train_xgb_numeric,  
        eval_set=[(X_val_processed, y_val_numeric)],
        verbose=False)
    y_val_pred = xgb.predict(X_val_processed)
    recall = recall_score(y_val_numeric, y_val_pred)
    # saves the recall (on val), model, and parameters
    if recall > best_recall:
        best_recall = recall
        best_xgb = xgb
        best_pram = pram

y_test_pred = best_xgb.predict(X_test_processed)

print(f'XGBoost\n')
print('Best params:', best_pram)
print('Best recall (val):', best_recall)
print(f'Best iteration: {best_xgb.best_iteration} (out of 500)\n')
print('Confusion Matrix: \n', confusion_matrix(y_test_numeric, y_test_pred))
print('Classification Report: \n', classification_report(y_test_numeric, y_test_pred))

XGBoost

Best params: {'learning_rate': 0.1, 'max_depth': 5}
Best recall (val): 0.8101604278074866
Best iteration: 19 (out of 500)

Confusion Matrix: 
 [[758 277]
 [ 73 301]]
Classification Report: 
               precision    recall  f1-score   support

           0       0.91      0.73      0.81      1035
           1       0.52      0.80      0.63       374

    accuracy                           0.75      1409
   macro avg       0.72      0.77      0.72      1409
weighted avg       0.81      0.75      0.76      1409



In [10]:
print('LR Recall (test):', recall_score(y_test, logreg_pipe.predict(X_test), pos_label='Yes'))
print('RF Recall (test):', recall_score(y_test, best_rf.predict(X_test), pos_label='Yes'))
print('XGB Recall (test):', recall_score(y_test_numeric, best_xgb.predict(X_test_processed), pos_label=1))

LR Recall (test): 0.5641711229946524
RF Recall (test): 0.7941176470588235
XGB Recall (test): 0.8048128342245989


In [34]:
# Gets predicted probabilities for positive class (churn = 'Yes' or 1)
probs_logreg = best_logreg.predict_proba(X_val)[:, 1]
probs_rf = best_rf.predict_proba(X_val)[:, 1]
probs_xgb = best_xgb.predict_proba(X_val_processed)[:, 1]

# Thresholds for logistic regression
print('Logistic Regression:')
for t in [.2, .25, .3, .35, .4, .45]:
    preds = (probs_logreg >= t).astype(int)
    rec = recall_score(y_val_numeric, preds)
    prec = precision_score(y_val_numeric, preds)
    print(f'  Threshold {t}: Recall={rec:.4f}, Precision={prec:.4f}')
print()

# Thresholds for random forest
print('Random Forest:')
for t in [.2, .25, .3, .35, .4, .45]:
    preds = (probs_rf >= t).astype(int)
    rec = recall_score(y_val_numeric, preds)
    prec = precision_score(y_val_numeric, preds)
    print(f'  Threshold {t}: Recall={rec:.4f}, Precision={prec:.4f}')
print()

# Thresholds for xgboost
print('XGBoost:')
for t in [.2, .25, .3, .35, .4, .45]:
    preds = (probs_xgb >= t).astype(int)
    rec = recall_score(y_val_numeric, preds)
    prec = precision_score(y_val_numeric, preds)
    print(f'  Threshold {t}: Recall={rec:.4f}, Precision={prec:.4f}')



Logistic Regression:
  Threshold 0.2: Recall=0.8476, Precision=0.4655
  Threshold 0.25: Recall=0.7995, Precision=0.5042
  Threshold 0.3: Recall=0.7513, Precision=0.5332
  Threshold 0.35: Recall=0.7032, Precision=0.5755
  Threshold 0.4: Recall=0.6578, Precision=0.6044
  Threshold 0.45: Recall=0.5909, Precision=0.6332

Random Forest:
  Threshold 0.2: Recall=0.9813, Precision=0.3663
  Threshold 0.25: Recall=0.9652, Precision=0.3963
  Threshold 0.3: Recall=0.9492, Precision=0.4236
  Threshold 0.35: Recall=0.9251, Precision=0.4488
  Threshold 0.4: Recall=0.8984, Precision=0.4699
  Threshold 0.45: Recall=0.8556, Precision=0.4893

XGBoost:
  Threshold 0.2: Recall=0.9626, Precision=0.3666
  Threshold 0.25: Recall=0.9492, Precision=0.3813
  Threshold 0.3: Recall=0.9305, Precision=0.4037
  Threshold 0.35: Recall=0.8930, Precision=0.4255
  Threshold 0.4: Recall=0.8663, Precision=0.4538
  Threshold 0.45: Recall=0.8396, Precision=0.4831


In [41]:
# Threshold on test data
rf_test_probs = best_rf.predict_proba(X_test)[:, 1]
xgb_test_probs = best_xgb.predict_proba(X_test_processed)[:, 1]

# if prob is greater than .45 returns as True
rf_test_preds = (rf_test_probs >= 0.45)
xgb_test_preds = (xgb_test_probs >= 0.45)

# compares prediction (above) to test
print('Random Forest:')
print(f'Recall:{recall_score(y_test_numeric, rf_test_preds)}')
print(f'Precision:{precision_score(y_test_numeric, rf_test_preds)}')

print('XGBoost:')
print(f'Recall:{recall_score(y_test_numeric, xgb_test_preds)}')
print(f'Precision:{precision_score(y_test_numeric, xgb_test_preds)}')


Random Forest:
Recall:0.8368983957219251
Precision:0.4860248447204969
XGBoost:
Recall:0.8342245989304813
Precision:0.4890282131661442


In [50]:
import joblib
final_threshold = 0.45
joblib.dump(best_xgb, '../models/xgb_model.pkl')
joblib.dump(preprocessor_xgb, "../models/preprocessor.pkl")
joblib.dump(final_threshold, "../models/threshold.pkl")

['../models/threshold.pkl']

Model Selection Rationale: 
I evaluated Logistic Regression, Random Forest, and XGBoost using recall as the primary metric due to the business cost of missing churned customers.

Random Forest and XGBoost achieved similar precision–recall trade-offs after threshold tuning

XGBoost was selected as the final model because it achieved very similar recall on the validation and test sets, offered better control over class imbalance using scale_pos_weight, and provided early stopping to reduce overfitting.